In [ ]:
%pip install ultralytics matplotlib opencv-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 3.8 MB/s eta 0:00:00


In [ ]:
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import json

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
INPUT_IMAGE = "horses-yolo.jpg"          # <- change to your image file path
OUTPUT_IMAGE = "output_annotated.jpg"
MODEL_NAME = "yolov8n.pt"         # yolov8n, yolov8s, yolov8m, etc.
CONF_THRESH = 0.25
IMGSZ = 640

In [ ]:
model = YOLO(MODEL_NAME)  # downloads weights automatically if needed
print("Loaded model:", MODEL_NAME)
print("Available classes (partial):", {k: v for k, v in list(model.names.items())[:10]})

Loaded model: yolov8n.pt
Available classes (partial): {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light'}


In [ ]:
results = model.predict(source=INPUT_IMAGE, imgsz=IMGSZ, conf=CONF_THRESH, save=False)
res = results[0]   # first (and only) image result

FileNotFoundError: horses-yolo.jpg does not exist

In [ ]:
import os
os.listdir()

['.config', 'yolov8n.pt', 'sample_data']

In [ ]:
# cell
detections = []
for box in res.boxes:
    # xyxy, conf, cls are available on each box
    xyxy = box.xyxy.cpu().numpy().tolist()[0]   # [x1, y1, x2, y2]
    conf = float(box.conf.cpu().numpy()[0]) if hasattr(box, "conf") else float(box.conf)
    cls_idx = int(box.cls.cpu().numpy()[0]) if hasattr(box, "cls") else int(box.cls)
    label = model.names.get(cls_idx, str(cls_idx))
    detections.append({
        "label": label,
        "class_id": cls_idx,
        "confidence": round(conf, 4),
        "x1": int(xyxy[0]),
        "y1": int(xyxy[1]),
        "x2": int(xyxy[2]),
        "y2": int(xyxy[3]),
    })

df = pd.DataFrame(detections)
if df.empty:
    print("No detections above confidence threshold.")
else:
    display(df)   # in Jupyter this shows a nice table

NameError: name 'res' is not defined

In [ ]:
# cell
try:
    annotated_rgb = res.plot()  # returns an RGB numpy array with drawn boxes/labels
    # convert RGB -> BGR for OpenCV saving
    annotated_bgr = cv2.cvtColor(annotated_rgb, cv2.COLOR_RGB2BGR)
    cv2.imwrite(OUTPUT_IMAGE, annotated_bgr)
    print("Annotated image saved to:", OUTPUT_IMAGE)

    # Show inline (convert back to RGB for matplotlib)
    img_show = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 10))
    plt.imshow(img_show)
    plt.axis("off")
    plt.show()

except Exception as e:
    print("Failed to auto-plot annotated image:", e)
    # Fallback: draw using OpenCV
    img = cv2.imread(INPUT_IMAGE)
    for det in detections:
        cv2.rectangle(img, (det["x1"], det["y1"]), (det["x2"], det["y2"]), (0,255,0), 2)
        cv2.putText(img, f'{det["label"]} {det["confidence"]:.2f}', (det["x1"], max(15, det["y1"]-5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 1)
    cv2.imwrite(OUTPUT_IMAGE, img)
    # show:
    img_show = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 10))
    plt.imshow(img_show)
    plt.axis("off")
    plt.show()

Failed to auto-plot annotated image: name 'res' is not defined


error: OpenCV(5.0.0) /io/opencv/modules/imgcodecs/src/loadsave.cpp:1174: error: (-215:Assertion failed) !_img.empty() in function 'imwrite'


In [ ]:
# cell
out_json = {"input": INPUT_IMAGE, "output": OUTPUT_IMAGE, "detections": detections}
with open("detection_results.json", "w") as f:
    json.dump(out_json, f, indent=2)
print("Saved detection_results.json")

Saved detection_results.json
